# Section 2: Feature Engineering & Predictive Modeling

This notebook loads **`amazon_cleaned.csv`** from the project root. If your file is named `cleaned_amazon.csv`, change `CSV_PATH` in the configuration cell.

**Contents**
- Feature engineering: review word count, VADER & TextBlob sentiment, price-drop percentage, categorical encodings, TF-IDF on `review_content`, and a **dense embedding** path (Word2Vec via Gensim when available, otherwise LSA / TruncatedSVD on TF-IDF as a stand-in).
- **Classification**: Logistic Regression, Decision Tree, and Random Forest for negative / neutral / positive labels derived from `rating`, plus evaluation.
- **Regression**: Linear Regression, Lasso, and Ridge to predict numeric `rating` with MSE, RMSE, and R².

In [1]:


from pathlib import Path
import os

NOTEBOOK_DIR = Path.cwd()

ROOT = (
    NOTEBOOK_DIR.parent
    if NOTEBOOK_DIR.name == "notebooks"
    else NOTEBOOK_DIR
)



DATA_RAW_DIR = ROOT / "data" / "raw"

DATA_PROCESSED_DIR = ROOT / "data" / "processed"

MODELS_DIR = ROOT / "models"

NOTEBOOKS_DIR = ROOT / "notebooks"

APP_DIR = ROOT / "app"


CSV_PATH = DATA_PROCESSED_DIR / "amazon_cleaned.csv"


DATA_RAW_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DATA_PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


RANDOM_STATE = 42


print("Current Working Directory:")
print(NOTEBOOK_DIR)

print("\nProject Root:")
print(ROOT)

print("\nDataset Path:")
print(CSV_PATH)

print("\nModels Directory:")
print(MODELS_DIR)

In [2]:
import subprocess
import sys

pkgs = ["vaderSentiment", "textblob", "nltk"]
# Gensim often lacks wheels on very new Python; install only when likely supported.
if sys.version_info < (3, 13):
    pkgs.append("gensim")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

0

In [3]:
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression, LinearRegression, Lasso, Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    mean_squared_error,
    r2_score,
)

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob

import nltk

nltk.download("punkt", quiet=True)
try:
    nltk.download("punkt_tab", quiet=True)
except Exception:
    pass

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 30)

## Load data

In [4]:
df = pd.read_csv(CSV_PATH)
print("shape:", df.shape)
df.head(3)

FileNotFoundError: [Errno 2] No such file or directory: '/content/amazon_cleaned.csv'

## Feature engineering

- **Review length**: word count in `review_content`.
- **Sentiment**: VADER `compound` and TextBlob `polarity` in [-1, 1].
- **Price drop %**: `(actual_price_inr - discounted_price_inr) / actual_price_inr * 100`, clipped to [0, 100].
- **Categorical encoding**: `category_primary` and `user_name` via sparse one-hot (frequent users capped).
- **TF-IDF** on `review_content` for the main sklearn pipelines. The next section adds **Word2Vec** (Gensim) or, if Gensim is unavailable, **LSA** (TruncatedSVD on TF-IDF) as a dense embedding baseline.

In [ ]:
text_col = "review_content"
df[text_col] = df[text_col].fillna("").astype(str)

df["review_length_words"] = df[text_col].str.split().str.len().clip(lower=0)

vader = SentimentIntensityAnalyzer()


def vader_compound(text: str) -> float:
    return float(vader.polarity_scores(text)["compound"])


def textblob_polarity(text: str) -> float:
    return float(TextBlob(text).sentiment.polarity)


df["sentiment_vader_compound"] = df[text_col].map(vader_compound)
df["sentiment_textblob_polarity"] = df[text_col].map(textblob_polarity)

ap = df["actual_price_inr"].astype(float)
dp = df["discounted_price_inr"].astype(float)
raw_pct = np.where(ap > 0, ((ap - dp) / ap) * 100.0, np.nan)
df["price_drop_pct"] = np.clip(raw_pct, 0, 100)
df["price_drop_pct"] = df["price_drop_pct"].fillna(df["price_drop_pct"].median())

df["category_primary"] = df["category_primary"].fillna("unknown").astype(str)
df["user_name"] = df["user_name"].fillna("unknown").astype(str)

num_feature_cols = [
    "review_length_words",
    "sentiment_vader_compound",
    "sentiment_textblob_polarity",
    "price_drop_pct",
]
df[num_feature_cols].describe()

### Sentiment labels (3-class from `rating`)

- **negative**: rating < 3
- **neutral**: 3 <= rating < 4
- **positive**: rating >= 4

This gives a neutral band for middling scores. The rubric’s binary rule (>=3 vs <3) is reported separately after training.

In [ ]:
def rating_sentiment_label(r: float) -> str:
    if r < 3:
        return "negative"
    if r < 4:
        return "neutral"
    return "positive"


df["sentiment_label_3"] = df["rating"].map(rating_sentiment_label)
print(df["sentiment_label_3"].value_counts())

df["sentiment_label_binary"] = (df["rating"] >= 3).astype(int)
print("\nbinary (>=3 positive):", df["sentiment_label_binary"].value_counts())

### Train / test split

In [ ]:
y_clf = df["sentiment_label_3"]
y_reg = df["rating"].astype(float)

X = df[[text_col, "category_primary", "user_name"] + num_feature_cols].copy()

X_train, X_test, y_train_clf, y_test_clf, y_train_reg, y_test_reg = train_test_split(
    X,
    y_clf,
    y_reg,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_clf,
)
print(X_train.shape, X_test.shape)

### Preprocessing: TF-IDF + numerics + one-hot

In [ ]:
preprocess_clf = ColumnTransformer(
    transformers=[
        (
            "tfidf",
            TfidfVectorizer(
                max_features=4000,
                ngram_range=(1, 2),
                min_df=2,
                sublinear_tf=True,
            ),
            text_col,
        ),
        ("num", StandardScaler(with_mean=False), num_feature_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), ["category_primary"]),
        (
            "user",
            OneHotEncoder(
                handle_unknown="ignore",
                max_categories=40,
                min_frequency=3,
                sparse_output=True,
            ),
            ["user_name"],
        ),
    ],
    remainder="drop",
    sparse_threshold=1.0,
)
preprocess_clf

## Sentiment classification

In [ ]:
log_reg = Pipeline(
    steps=[
        ("prep", preprocess_clf),
        (
            "clf",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                solver="saga",
                n_jobs=-1,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

tree_clf = Pipeline(
    steps=[
        ("prep", preprocess_clf),
        (
            "clf",
            DecisionTreeClassifier(
                max_depth=12,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

rf_clf = Pipeline(
    steps=[
        ("prep", preprocess_clf),
        (
            "clf",
            RandomForestClassifier(
                n_estimators=200,
                class_weight="balanced",
                n_jobs=-1,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

clf_models = {
    "Logistic Regression": log_reg,
    "Decision Tree": tree_clf,
    "Random Forest": rf_clf,
}

for name, model in clf_models.items():
    model.fit(X_train, y_train_clf)
    pred = model.predict(X_test)
    print("=" * 60)
    print(name)
    print("accuracy:", round(accuracy_score(y_test_clf, pred), 4))
    labels = ["negative", "neutral", "positive"]
    print("confusion matrix (rows=true, cols=pred):")
    print(pd.DataFrame(confusion_matrix(y_test_clf, pred, labels=labels), index=labels, columns=labels))
    print(classification_report(y_test_clf, pred, labels=labels, zero_division=0))

### Binary rubric check (predictions mapped to >=3 vs <3)

In [ ]:
y_test_bin = (y_test_reg >= 3).astype(int)

for name, model in clf_models.items():
    pred3 = model.predict(X_test)
    pred_bin = np.where(pred3 == "negative", 0, 1)
    print(name, "binary accuracy:", round(accuracy_score(y_test_bin, pred_bin), 4))

## Dense text vectors: Word2Vec (Gensim) or LSA fallback

**Word2Vec** (skip-gram, mean-pooled) when `gensim` imports. On Python versions without a working Gensim wheel, we use **TruncatedSVD** on TF-IDF (latent semantic / LSA-style dense vectors) plus the same numeric features, then multinomial Logistic Regression for a fair comparison to the full TF-IDF pipeline.

In [ ]:
USE_W2V = False
try:
    from gensim.models import Word2Vec

    USE_W2V = True
except Exception as e:
    print("Gensim Word2Vec unavailable:", e)

if USE_W2V:
    sentences = [row.split() for row in X_train[text_col].tolist()]
    w2v = Word2Vec(
        sentences=sentences,
        vector_size=100,
        window=5,
        min_count=2,
        workers=4,
        sg=1,
        epochs=15,
        seed=RANDOM_STATE,
    )

    def doc_vector(text: str) -> np.ndarray:
        vecs = [w2v.wv[w] for w in text.split() if w in w2v.wv]
        if not vecs:
            return np.zeros(w2v.vector_size)
        return np.mean(vecs, axis=0)

    X_train_emb = np.vstack(X_train[text_col].map(doc_vector).to_numpy())
    X_test_emb = np.vstack(X_test[text_col].map(doc_vector).to_numpy())
    emb_name = "Word2Vec mean-pool (100d)"
else:
    vec = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    Xtr = vec.fit_transform(X_train[text_col])
    Xte = vec.transform(X_test[text_col])
    n_comp = min(100, Xtr.shape[1] - 1, Xtr.shape[0] - 1)
    n_comp = max(2, n_comp)
    svd = TruncatedSVD(n_components=n_comp, random_state=RANDOM_STATE)
    X_train_emb = svd.fit_transform(Xtr)
    X_test_emb = svd.transform(Xte)
    emb_name = f"TruncatedSVD on TF-IDF ({n_comp}d, LSA-style)"

num_tr = X_train[num_feature_cols].to_numpy(dtype=float)
num_te = X_test[num_feature_cols].to_numpy(dtype=float)
scaler_emb = StandardScaler(with_mean=True)
num_tr_s = scaler_emb.fit_transform(num_tr)
num_te_s = scaler_emb.transform(num_te)

X_train_full = np.hstack([X_train_emb, num_tr_s])
X_test_full = np.hstack([X_test_emb, num_te_s])

lr_emb = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    solver="lbfgs",
    random_state=RANDOM_STATE,
)
lr_emb.fit(X_train_full, y_train_clf)
pred_emb = lr_emb.predict(X_test_full)
print(emb_name)
print("accuracy:", round(accuracy_score(y_test_clf, pred_emb), 4))
print(classification_report(y_test_clf, pred_emb, zero_division=0))

## Regression: predict rating (MSE, RMSE, R²)

In [ ]:
preprocess_reg = ColumnTransformer(
    transformers=[
        (
            "tfidf",
            TfidfVectorizer(
                max_features=4000,
                ngram_range=(1, 2),
                min_df=2,
                sublinear_tf=True,
            ),
            text_col,
        ),
        ("num", StandardScaler(with_mean=False), num_feature_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), ["category_primary"]),
        (
            "user",
            OneHotEncoder(
                handle_unknown="ignore",
                max_categories=40,
                min_frequency=3,
                sparse_output=True,
            ),
            ["user_name"],
        ),
    ],
    remainder="drop",
    sparse_threshold=1.0,
)

reg_models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(alpha=1e-3, max_iter=5000, random_state=RANDOM_STATE),
    "Ridge": Ridge(alpha=1.0, random_state=RANDOM_STATE),
}

rows = []
for name, est in reg_models.items():
    pipe = Pipeline([("prep", preprocess_reg), ("reg", est)])
    pipe.fit(X_train, y_train_reg)
    pred = pipe.predict(X_test)
    mse = mean_squared_error(y_test_reg, pred)
    rmse = float(np.sqrt(mse))
    r2 = r2_score(y_test_reg, pred)
    rows.append({"model": name, "MSE": mse, "RMSE": rmse, "R2": r2})

pd.DataFrame(rows).round({"MSE": 5, "RMSE": 5, "R2": 4})

### Ridge residuals (optional)

In [ ]:
ridge_pipe = Pipeline([("prep", preprocess_reg), ("reg", Ridge(alpha=1.0, random_state=RANDOM_STATE))])
ridge_pipe.fit(X_train, y_train_reg)
pred_ridge = ridge_pipe.predict(X_test)
resid = y_test_reg.to_numpy() - pred_ridge
pd.Series(resid).describe()